### Load packages

Kernel: if run locally use .conda 3.8

In [3]:
import torch
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime
import math

torch.set_printoptions(sci_mode = False)

ImportError: /home/kim/super-resolution/.conda/lib/python3.8/site-packages/zmq/backend/cython/../../../../.././libstdc++.so.6: version `GLIBCXX_3.4.30' not found (required by /home/kim/super-resolution/.conda/lib/python3.8/site-packages/torch/lib/libtorch_python.so)

In [4]:
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

NameError: name 'torch' is not defined

### T0DO:
- NLL for batches
- Check LML: Are we using ground truth?
- LML Hyperparameter optimisation
  - repeat per batch

- Run analytics on Google Colab to check bottlenecks

#### Experiments:
- Spatial kernel
- Compositite kernel
  - w/o pixel correction
- Gradient kernel
  - similarity between lr bed elevation gradients and aux gradients
- Global hyperparameters
  - Tune hyperparameters separetly for various upscaling factors

## Load data

In [ ]:
# 34 MB data
training_tensor = torch.load(
    './torch_data/TRANSANT_experiments/TRANSANT_training_tensor.pt')
# Subset first two channels
training_tensor = training_tensor[:, (0, 1), :, :]

# testing_tensor = torch.load(
# './torch_data/TRANSANT_experiments/TRANSANT_testing_tensor.pt')

# Dome C
# training_tensor = torch.load(
# '/Users/kimbente/super-resolution/torch_data/DOMEC_bed_scenes_60pixel.pt')

## Define class with functions

ToDos:
- finish documentation 
- DataLoaders to avoid kernel from crashing by using batches within functions
- Easy way of running partial kernel(spatial kernel)

In [ ]:
# from tensordict.tensordict import TensorDict
# TensorDict({})

In [ ]:
class Superresolution_GP:
    # Initialise object only over overall dimensionalitie
    def __init__(self, hr_hw, up_factors, initial_hyperparameters, device):
        """_summary_

        Args:
    
            hr_hw (torch.tensor(size = [1])): 1d torch_tensor containing the dimensionality of the target/ground truth scenes. 
                Scenes are always square with H == W. Must not be on device yet, as this will be initialised.
            up_factors (torch.tensor(size = [n_up_factors])): 1d torch tensor containing all upscaling factors (magnification factors) which will be applied.
            initial_hyperparameters(torch.tensor(size = [3, 1])): lambda_s, lambda_p, lambda_f, 
                must be .to(torch.float)
            device (device object): device(type='cuda')

        Raises:
            ValueError: Error if the upscaling_factors don't perfectly match hr_hw.
        """
        self.hr_hw = hr_hw.to(device) # High-resolution W (width) which is equal to H: H_h, W_h. torch.Size([1])
        self.up_factors = up_factors.to(device)  # torch.Size([5])
        # Extract number of up_factors
        self.n_up_factors = torch.tensor(self.up_factors.shape, device = device)
        # Low-resolution HW for each different up-factor, torch.Size([5]), integers, H_l, W_l.
        self.up_lr_hw = (self.hr_hw.repeat(self.n_up_factors) / self.up_factors).to(torch.int)

        # Create meta aggregation dictionary once, as global variable
        self.up_lr_to_hr_dict = self.make_aggregation_dictionaries()

        # CHECKPOINT: Test if self.hr_hw can be divided by ALL upscaling factors without a remainder. 
        # Check because this implementation only applies to subset cases.
        if torch.all(self.up_lr_hw.to(torch.float).isclose(self.hr_hw.repeat(self.n_up_factors) / self.up_factors)) == False:
            raise ValueError("Upscaling factors and Height/Width of pixel are not compatible. Use a different algorithm or a subset of compatible upscaling factors.")    

        # Hyperparameters initialisation
        self.gp_hyper_lambda_s = initial_hyperparameters[0].to(device).requires_grad_(True)
        self.gp_hyper_lambda_p = initial_hyperparameters[1].to(device).requires_grad_(True)
        self.gp_hyper_sigma_f = initial_hyperparameters[2].to(device).requires_grad_(True)# amplitude

        # Noise and constant mean, not treated as hyperparameters currently
        self.gp_hyper_noise = torch.tensor([0.05], device = device)
        self.gp_mu = torch.tensor([0.5], device = device)

    #############
    ### UTILS ###
    #############

    def upscale(self, bed_ground_truth, device = device):
        """ Upscale (increase scale of each pixel, reduce resolution) to articially generate low-resolution input. 
        Controlled experiment.

        Args:
            bed_ground_truth (torch.tensor(size = [N, C, H_h, W_h]) with C = 1): ground truth tensor at high resolution
                e.g. torch.Size([300, 1, 60, 60])

        Returns:
            dictionary of len n_up_factors where each entry is a torch.tensor(size = [N, C, H_l, W_l]) with C = 1. 
                The up_factors are the keys
        """
        # Initialize and empty dictionary on device
        up_train_lr = dict(device = device)
        # Iterate through upscaling factors
        for u in self.up_factors:
            # define upscaling function with torch https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html, 
            # Input is (N, C, H_h, W_h) and output is (N, C, H_l, W_l). default settings
            upscaling_function = torch.nn.AvgPool2d(kernel_size = u.item())
            # Add entry to dictionary
            up_train_lr[u.item()] = upscaling_function(bed_ground_truth)
        
        # DICT can't be put on cuda
        return up_train_lr
    
    def set_hypers(self, new_hyperparameters):
        """Update hyperparameters stored in object.

        Args:
            new_hyperparameters (torch.tensor(size = [3, 1])): lambda_s, lambda_p, lambda_f, 
                must be .to(torch.float)): _description_
        """
        self.gp_hyper_lambda_s = new_hyperparameters[0].to(device).requires_grad_(True)
        self.gp_hyper_lambda_p = new_hyperparameters[1].to(device).requires_grad_(True)
        self.gp_hyper_sigma_f = new_hyperparameters[2].to(device).requires_grad_(True)

    def make_aggregation_dictionaries(self):
        """Generates aggregation dictionaries once.

        Returns:
            dict(): Dictionary of dictionaries. 
            meta-level keys are up_factors (u.itemn())
        """

        # Initialise meta_dictionary to hold aggregation dictionary
        # for each up_factor
        up_lr_to_hr_dict = {}

        # u_index is an index int and u is a tensor. u.item() returns the int
        for u_index, u in enumerate(self.up_factors):

            # print(u_index)
            # print(u.item())

            # retrieve lr_hw tensor for the respective u from self
            lr_hw = self.up_lr_hw[u_index].to(device)
            # create aggregation (lr) target indices (e.g. 0 to 899)
            # covariance matrix has squared indices (pairwise)
            lr_flat_indices = torch.arange(
                start =  0, 
                end = lr_hw**2, 
                device = device)
            
            # Indicate row breaks of lr w.r.t. the hr covariance matrix
            # (e.g. 0, 120, 240) going from 30**2 to 60**2
            # includes 0 and last index and intevals are u * hr_hw wide
            lr_row_breaks = torch.linspace(
                start = 0, 
                end = self.hr_hw.item()**2, 
                steps = int(lr_hw + 1), 
                dtype = int, 
                device = device)
            
            # create empty dictionary that maps every lr index 
            # to the corresponding hr indices
            lr_to_hr_dict = dict()

            # start with -1 as it will be updated in first iteration
            lr_row_counter = -1
            
            # interate over all lr covariance indices to create the 
            # dictionary entry
            for i in lr_flat_indices:
                # column counter per row
                j = i % lr_hw

                # check for row breaks
                if j == 0:
                    # Update in very first iteration 
                    lr_row_counter += 1
                    row_base = lr_row_breaks[lr_row_counter]

                # generate list of hr_indices that correspond to each i/j
                corresponding_hr_indices = []


                for k in range(u.item()):
                    # each list is spanning upscaling_factor x rows with 
                    # upscaling_factor x elements each
                    corresponding_hr_indices.extend(
                        range((j * u + (k * self.hr_hw) + row_base),
                            (j * u + (k * self.hr_hw) + u + row_base)))

                lr_to_hr_dict[i.item()] = corresponding_hr_indices

            # Assign dictionary to meta-dictionary for every up_factor   
            up_lr_to_hr_dict[u.item()] = lr_to_hr_dict

        return up_lr_to_hr_dict
    
    #############
    ### BATCH ###
    #############

    def predict_batch(self, batch):
        """Prediction over batches.

        Args:
            batch (torch.tensor(size = [N, 2, H_h, W_h]) where N is the batch size): two channel
                batch_size may change with data_loader iteration

        Returns:
            _type_: _description_
        """
        # Extract bed elevation channel: [N, C, H_h, W_h]. unsqueeze() retains explicit channel dimension
        train_bed_ground_truth = batch[:, 0, :, :].unsqueeze(1).to(device) # channel 0 contains bed elevation

        # Extract auxiliary channel [N, C, H_h, W_h]
        train_sur_hr_aux = batch[:, 1, :, :].unsqueeze(1).to(device) # channel 1 contains surface elevation

        # Upscale to create dictionary holding low resolution input tensors [N, C, H_l, W_l]
        # DICT can't be put on cuda
        train_bed_lr = self.upscale(
            train_bed_ground_truth, device = device)

        ### BASE COVARIANCE ###
        # Generate base covariance_matrices using currently stored hypers
        # Input: train_sur_hr_aux (spatial is arbitrary)
        k_ah_ah = self.composite_base_covariance(
            train_sur_hr_aux, device = device).to(device)

        ### AGGREGATE ###
        # Pass in batch of base covariances
        k_ah_al, k_al_al = self.aggregate_base_covariance(
            k_ah_ah, device = device)

        ### PREDICTIVE ###
        # TODO: LML needs to use ground truth?!
        batch_mean, batch_covariance, batch_lml = self.predictive_distribution(
            k_ah_ah, k_ah_al, k_al_al, train_bed_lr, device)

        return batch_mean, batch_covariance, batch_lml
    
    #################
    ### BASELINE ####
    #################

    def bilinear_interpolation_baseline(self, bed_lr):
        """Baseline method

        Args:
            bed_lr (list of torch.tensors): list of torch tensors of different dimensionalities
        """
        # bed_lr is a list of tensors

        # Create one target grid: outer boundries of each scene are the same for both hr and lr: [-1, 1]
        d = torch.tensor(np.linspace(start = (-1.0 + (2/self.hr_hw)/2) , stop = (1.0 - (2/self.hr_hw)/2), num = self.hr_hw))
        meshx, meshy = torch.meshgrid((d, d), indexing = "xy") # create mesh
        target_grid = torch.stack((meshx, meshy), 2) # x,y order
        target_grid = target_grid.unsqueeze(0) # add batch dim: torch.Size([1, self.hr_hw, self.hr_hw, 2])

        # Create empty placeholder tensor of shape [Up = 0, N, C = 1, H, W]
        up_n_bed_hr = torch.empty(size = (0, bed_lr[0].shape[0], 1, self.hr_hw, self.hr_hw))

        # Simple baseline in bilinear interpolation using torch https://pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html 
        for u_index, u in enumerate(self.up_factors):

            # copy target grid n times
            n = bed_lr[u_index].shape[0]
            n_target_grids = torch.tile(target_grid, dims = (n, 1, 1, 1))

            hr_bilinear = torch.nn.functional.grid_sample(bed_lr[u_index].float(), n_target_grids.float(), mode = 'bilinear', padding_mode = 'border', align_corners = False)
            up_n_bed_hr = torch.cat((up_n_bed_hr, hr_bilinear.unsqueeze(0)), dim = 0) # generate explicit first dim for all up_factors

        return(up_n_bed_hr) # [Up, N, C = mean, H, W]
    
    ###############
    ### METRICS ###
    ###############

    def rmse(self, ground_truth, predictions):
        """_summary_

        Args:
            ground_truth (_type_): [N, 1, H, W] - will be copied for all up_factors
            predictions (_type_): [Up, N, 1, H, W]

        Returns:
            [Up, N, 1]
        """
        n = predictions.shape[0]
        # Copy for n_Up_factors into [Up, N, 1, H, W]
        n_ground_truth = torch.tile(ground_truth.unsqueeze(0), dims = (n, 1, 1, 1, 1))

        error = torch.sub(n_ground_truth, predictions) # subtract elementwise
        squared_error = torch.pow(error, exponent = 2) # square error to eliminate negatives
        mean_squared_error = torch.mean(squared_error, dim = (-2, -1)) # mean across H and W (last two dim)
        root_mean_squared_error = torch.sqrt(mean_squared_error)
        
        # ToDo: mean over N
        return root_mean_squared_error # [Up, N, C = RMSE]
    
    def nll(self):
        # TBC
        return 0
    
    ##################
    ### COVARIANCE ###
    ##################

    def composite_base_covariance(self, batch_aux_tensor, device):
        """Takes in a batch of aux_tensors and produces the pairwise covaraince matricex for the batch.
        The spatial base covariance part of the product compositive kernel is the same for all N thus we are creating N copies for all batch members. 
        The pixel base covariance is calculated over batches. 

        Args:
            batch_aux_tensor (torch.tensor(size = [N, C, H_h, W_h])): with N = batch_size and C is one channel

        Returns:
            torch.tensor(size = [N, C, H_h **2, W_h ** 2]): base covariance matrix k_ah_ah
        """
        # spatial_base_covariance does not depend on any inputs

        # Pass batch size into spatial covariance function
        spatial_base_covariance_matrix = self.spatial_base_covariance(batch_aux_tensor.shape[0], device).to(device)
        pixel_base_covariance_matrix = self.pixel_base_covariance(batch_aux_tensor, device).to(device)
            
        # Element-wise multiplication, Hadamard product, AND operation
        product_base_covar = torch.mul(spatial_base_covariance_matrix, pixel_base_covariance_matrix).to(device)
            
        # multiply all elements in matrix with same scalar, in-place
        product_base_covar = torch.mul(product_base_covar, self.gp_hyper_sigma_f).to(device)

        return product_base_covar
    
    def only_spatial_base_covariance(self):
        # Fast do no batching needed
        spatial_base_covariance_matrix = self.spatial_base_covariance(self.n_train)

        return spatial_base_covariance_matrix

    def spatial_base_covariance(self, n_batch, device):
        """ Calculate spatial covariance using n_batches, self.hr_hw and self.gp_hyper_lambda_s for the scaling.
        Smooth, sparse and local.
        Lambda_s is the hp that controls the receptive field.
        Same for all n so we calculate it once and create copies.

        Args:
            n_batch (int): number of copies we need
            device (object): cuda or cpu

        Returns:
            torch.tensor(size = [N, C, H_h ** 2, W_H ** 2]): with N = n_batches and C = 1 which is the covariance channel.
        """
        # Normalised mid_points of all pixels in scene
        xs = torch.arange(0, (self.hr_hw.item())).repeat(self.hr_hw.item(), 1).to(device)
        ys = xs.mT.to(device)
        # x and y dim of midpoints from 0 to 1 
        mid_points_norm = torch.cat((ys.unsqueeze(0), xs.unsqueeze(0)), dim = 0)/(self.hr_hw - 1).to(device) # torch.Size([2, 60, 60])

        # Flatten the last two dims
        mid_points_flat = torch.flatten(mid_points_norm, start_dim = -2).to(device) # torch.Size([2, 3600])
        # broadcast to calculate pairwise distance
        dist = torch.sub(mid_points_flat.unsqueeze(-1), mid_points_flat.unsqueeze(-2)).to(device)
        # square all distances
        dist_square = torch.pow(dist, exponent = 2).to(device)
        # sum x direction dist and y direction dist (Euclidean dist)
        dist_sum = torch.sum(dist_square, dim = 0).to(device)
        dist_euc = torch.sqrt(dist_sum).to(device) # torch.Size([3600, 3600]), Pythagoras, max is 1.4142 (corners, sqrt(2))

        Z = torch.div(dist_euc, self.gp_hyper_lambda_s).to(device) # divide by scalar, lambda_s is threshold for 0 covariance
        # Mask large Z's (too-far-away values) with nan before replacing values with zero later
        Z[Z >= 1] = float('nan')
        # first term pushes small distances to 0 and distances near 1 close to zero
        # second terms is clipped at 1 so that small distances will approach 1
        prod_term1 = torch.pow(torch.add(-Z, 1.), exponent = 3).to(device) # (1 - Z) = (-Z + 1)
        prod_term2 = torch.add(torch.mul(Z, 3.), 1.).to(device)
        cov_matrix = torch.mul(prod_term1, prod_term2).to(device) # elementwise multiplication

        # fill nan's with zero
        cov_matrix[torch.isnan(cov_matrix)] = 0.0
        cov_matrix = cov_matrix.unsqueeze(0).unsqueeze(1) # torch.Size([1, 1, 3600, 3600])

        # Create n copies
        n_cov_matrix = torch.tile(cov_matrix, dims = (n_batch, 1, 1, 1)).to(device) # torch.Size([n, 3600, 3600])

        # torch.Size([5, 1, 3600, 3600])
        return n_cov_matrix

    def pixel_base_covariance(self, batch_aux_tensor, device):
        """coupling(decoupling) of similar(dissimilar) pixel values, non-stationary
        lambda_p is the lengthscale of the RBF kernel, default is 0.6931 in GPytorch
        for 300 scenes this has a wall time of 8:20 min on MacPro
        Currently handles full batch at a time.
        Overwriting of variable names to reduce memory.

        Args:
            batch_aux_tensor (torch.tensor(size = [N, C, H_h, W_h])): a batch of the auxiliary variable
            device (object): cuda or cpu

        Returns:
            torch.tensor(size = [N, C, H_h **2, W_h **2]): pixel covaraince
        """
        # Flatten x & y / reduce the last two dimensions and overwrite
        batch_aux_tensor = torch.flatten(batch_aux_tensor, start_dim = -2).to(device)

        ### Do element-wise if it crashes ###

        # Overwriting for memory
        # DISTANCE: broadcast for pairwise dist (in pixel dim): torch.Size([5, 1, 3600, 1]), torch.Size([5, 1, 1, 3600])
        dist = torch.sub(batch_aux_tensor.unsqueeze(-1), batch_aux_tensor.unsqueeze(-2)).to(device)

        # DISTANCE SQUARED: direction of distance does not matter, overwrite for memory
        dist = torch.pow(dist, exponent = 2)
        # DISTANCE SQUARED AND SCALED: 
        dist = torch.div(dist, (2 * torch.pow(self.gp_hyper_lambda_p, exponent = 2)))

        # Exponent will evaluate to 1 if there is zero distance
        rbf = torch.exp( - dist).to(device)
        return rbf
    
    def aggregate_base_covariance(self, batch_base_covariance_matrix, device):
        """Aggregate k_ah_ah to k_ah_al (column aggregation) and then k_al_al
        using the global aggregation dictionary created once at 
        initialisation of object.

        Args:
            batch_base_covariance_matrix (torch.tensor(size = [N, C, H_h, W_h])): 
                k_ah_ah
            device (object): cuda or cpu

        Returns:
            dict: Dictionary with a torch.tensor(size = [N, 1, H_h, W_l]) 
                  for each up_factor. Tensors on device.
            dict: Dictionary with a torch.tensor(size = [N, 1, H_l, W_l]) 
                  for each up_factor. Tensors on device.
        """

        # Intialise empty dictionaries which will have one tensor entry per up_factor
        up_k_ah_al = dict()
        up_k_al_al = dict()

        for u_index, u in enumerate(self.up_factors):
            # use nested list of all dictionary entries for respective up_factor
            # to subset columns of covariance matrix.
            # Simple subset: average over last dimension (equally weighted). 
            # cardinality of last dim changes: for up = 2: average over 4 values
            up_k_ah_al[u.item()] = torch.mean(
                batch_base_covariance_matrix[:, :, :, list(self.up_lr_to_hr_dict[u.item()].values())], 
                dim = -1).to(device)
            
            # apply .mt to transpose the last two dims and repeat
            up_k_al_al[u.item()] = torch.mean(
                up_k_ah_al[u.item()].mT[:, :, :, list(self.up_lr_to_hr_dict[u.item()].values())], 
                dim = -1).to(device)
            
        return up_k_ah_al, up_k_al_al
    
    ##################
    ### PREDICTIVE ###
    ##################
    
    def predictive_distribution(self, k_ah_ah, k_ah_al, k_al_al, train_lr, device):
        """Calculation of the mean and variance of the Gaussian predictive distribution as well as of Log Marginal Likelihood. 
        Here we implement the inversion through the Cholesky decomposition [torch.linalg.cholesky()] followed by the inversion [torch.cholesky_inverse(L)]. 
        This is slighly different to the Algorithm 2.10 proposed in Rasmussen & Williams but more suitable to implement the weight correction performed by Reid et al.
        - Loop over up_factors, but operate on batch at once.

        k_ah_ah is a torch tensor.
        k_ah_al is a list of tensors.
        k_al_al is a list of tensors.
        train_lr is a 
        """
        
        # Go through one at a time
        # n_base_covars = k_ah_ah.shape[0]

        # empty tensor
        up_n_mean = torch.empty(size = (0, k_ah_ah.shape[0], 1, self.hr_hw, self.hr_hw), device = device)
        up_n_covariance = torch.empty(size = (0, k_ah_ah.shape[0], 1, self.hr_hw**2, self.hr_hw**2), device = device)
        up_n_lml = torch.empty(size = (0, k_ah_ah.shape[0]), device = device)

        for u_index, u in enumerate(self.up_factors):
            # both L and k_inv are used throughout
            L = torch.linalg.cholesky(torch.tile((torch.eye(n = k_al_al[u.item()].shape[-1], device = device) * self.gp_hyper_noise).unsqueeze(0).unsqueeze(0), dims = [k_al_al[u.item()].shape[0], 1, 1, 1]))
            k_inv = torch.cholesky_inverse(L).to(device)

            #### Mean ###
            pl_al_minus_mu = torch.sub(train_lr[u.item()], 
                                       torch.mul(torch.ones(size = train_lr[u.item()].shape, device = device), 
                                                 self.gp_mu)).to(device)
            W = torch.matmul(k_ah_al[u.item()], k_inv).to(device) 

            # Correction, torch.div is element-wise vision
            # batched multiplication: torch.Size([6, 1, 3600, 900]) torch.Size([6, 1, 900, 1])
            # print(torch.matmul(W, torch.flatten(pl_al_minus_mu, start_dim = -2).unsqueeze(-1)).shape) torch.Size([6, 1, 3600, 1])
            # ones: torch.Size([900, 1]) torch.Size([6, 1, 3600, 1])
            n_mean = torch.div(
                torch.matmul(W, torch.flatten(pl_al_minus_mu, start_dim = -2).unsqueeze(-1)), 
                torch.matmul(W, torch.ones(size = (W.shape[-1], 1), device = device))) + self.gp_mu
            # cast into 2D shape: N, C, H, W
            n_mean = n_mean.reshape(n_mean.shape[0], 1, int(np.sqrt(n_mean.shape[- 2])), -1)

            ### Covariance ###
            # .mT transposes the last two dims of a matrix
            n_covariance = k_ah_ah - torch.matmul(k_ah_al[u.item()], 
                                                  torch.matmul(k_inv, k_ah_al[u.item()].mT))

            ### LML ###
            # 2.30 in Rasmussen
            # https://d2l.ai/chapter_gaussian-processes/gp-inference.html
    
            # Term1: Kernel term
            term1 = torch.mul(torch.matmul(torch.flatten(train_lr[u.item()].mT, start_dim = -2).unsqueeze(-2), torch.matmul(k_inv, torch.flatten(train_lr[u.item()].mT, start_dim = -2).unsqueeze(-1))), 0.5)

            # Term2: Determinant term. 2 and 0.5 cancel each other out, sum in log space, log makes values negative
            term2 = torch.sum(torch.log(torch.diagonal(L, dim1 = -2, dim2 = -1)), dim = (1, 2)).to(device)

            # Need trick https://math.stackexchange.com/questions/3158303/using-cholesky-decomposition-to-compute-covariance-matrix-determinant
            # Does not work: term2 = torch.mul(torch.log(torch.linalg.det(k_al_al + (torch.eye(n = k_al_al.shape[-1]) * noise))), 0.5).reshape(1, 1)
            # Flatten shape and extract n, natural log
    
            # Term3: Constant term: Only extracts shape from train_lr
            term3 = (torch.log(torch.tensor(2 * math.pi, device = device)) * torch.flatten(train_lr[u.item()], start_dim = - 2).shape[-1] * 0.5).reshape(-1)
            n_lml = (- term1.reshape(-1) - term2 - term3).to(device)

            up_n_mean = torch.cat((upfactors_n_mean, n_mean.unsqueeze(0)), dim = 0)
            up_n_covariance = torch.cat((upfactors_n_covariance, n_covariance.unsqueeze(0)), dim = 0)
            up_n_lml = torch.cat((upfactors_n_lml, n_lml.unsqueeze(0)), dim = 0)

        return up_n_mean, up_n_covariance, up_n_lml


In [ ]:
hr_hw = torch.tensor([60])
up_factors = torch.tensor([2, 3, 4, 5, 6])
initial_hyperparameters = torch.tensor([[0.4], [0.7], [1.0]]).to(torch.float)

srGP = Superresolution_GP(hr_hw, up_factors, initial_hyperparameters, device)

training_batch = training_tensor[0:6, :, :, :]

upfactors_n_mean, upfactors_n_covariance, upfactors_n_lml = srGP.predict_batch(training_batch)

In [ ]:
def scale(input_tensor):
    """Min-max normalisation that projects inputs into [0, 1] (inclusive, inclusive) range.
    Can be turned off using self.scaling boolean.
    Currently only implemented to work on 1-Channel inputs.

    Args:
        input_tensor (torch.tensor): input tensor

    Returns:
        (torch.tensor): scaled version of input. 
    """
    # Update global parameter
    scaling_global_min = torch.min(input_tensor)
    scaling_global_max = torch.max(input_tensor)
    scaling_global_range = (scaling_global_max - scaling_global_min)

    minmax_scaled = (input_tensor - scaling_global_min) / scaling_global_range

    # Optional: save global values to self to apply same standardisation for test
    return minmax_scaled, scaling_global_range, scaling_global_min

# Predictions

In [ ]:
# Overwrite with scales version
scaled_training_tensor = training_tensor
scaled_training_tensor[:, 0, :, :], scaling_global_range, scaling_global_min = scale(training_tensor[:, 0, :, :])
scaled_training_tensor[:, 1, :, :], aux_scaling_global_range, aux_scaling_global_min  = scale(training_tensor[:, 1, :, :])

scaled_training_tensor[:, 0, :, :]
print("Scaling range: {} Scaling minimum: {}".format(np.round(scaling_global_range.item(), decimals = 2), np.round(scaling_global_min.item(), decimals = 2)))

# RMSE loop

In [ ]:
torch.device('cuda:0')

In [ ]:
# Initialise object
srGP_full = SuperresolutionGP_lowerRAM(train_tensor = scaled_training_tensor)

# Calc baseline mean prediction, mean over all scenes
baseline_rmse = torch.mean(srGP_full.rmse(srGP_full.train_bed_ground_truth, srGP_full.bilinear_interpolation_baseline(srGP_full.train_bed_lr)), dim = (1, 2)) # [Up, N, C = RMSE]
print("Baseline RMSE in meters: {}". format((baseline_rmse * scaling_global_range.repeat((srGP_full.n_up_factors, )))))
print("Baseline RMSE scaled: {}". format(baseline_rmse))

# Empty tensor for rsme's
n_rmse = torch.empty(size = (srGP_full.n_up_factors, 0, 1), device = device) # [Up, N = 0, C]

# Apply batchwise so it does not crash
outer_dataloader = torch.utils.data.DataLoader(scaled_training_tensor, batch_size = 10, shuffle = False)
for batch in outer_dataloader:
    srGP = SuperresolutionGP_lowerRAM(train_tensor = batch.to(device))
    srGP.to(device)
    # return [Up, N, C = 1]
    batch_rmse = srGP.predict_and_rmse()
    n_rmse = torch.cat((n_rmse, batch_rmse), dim = 1) # dim 1
    print("Cumulative batch mean:", torch.mean(n_rmse, dim = (1, 2)))

proposed_rmse = torch.mean(n_rmse, dim = (1, 2))

print("Proposed RMSE in meters: {}". format((proposed_rmse * scaling_global_range.repeat((srGP_full.n_up_factors, )))))
print("Proposed RMSE scaled: {}". format((proposed_rmse)))

In [ ]:
def visualise_results(baseline_rmse_tensor, proposed_rmse_tensor, n_scenes, domain_name):

    # RMSE
    fig = go.Figure()
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = baseline_rmse_tensor, mode = 'lines+markers', name = "Bilinear baseline"))
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_rmse_tensor, mode = 'lines+markers', name = "Proposed algorithm"))
    fig.update_layout(title = 'Reconstruction loss [RMSE] of proposed vs. baseline - {} scenes near domain {}'.format(n_scenes, domain_name), 
    template = "plotly_white")
    fig.update_xaxes(title_text = 'Upscaling factor')
    fig.update_yaxes(title_text = 'RMSE')
    fig.show()

In [ ]:
visualise_results(baseline_rmse, proposed_rmse, n_scenes = 300, domain_name = "Transantarctic mountains")


In [ ]:
lr_hw = 60
torch.arange(0, lr_hw**2)

np.linspace(0, lr_hw**2, num = int(lr_hw + 1), dtype = int)

torch.linspace(start = 0, end = lr_hw**2, steps = lr_hw + 1)

In [ ]:
dic = dict()
dic[1] = 23
dic

In [ ]:
proposed_rmse

300 images, 2 upscale factors: 6 minutes

I want RMSE, NLL, LML
# Visualisation function

# All outputs in one loop

In [ ]:
dataloader = torch.utils.data.DataLoader(scaled_training_tensor[:, :, :, : ], batch_size = 5, shuffle = False)
n_up = 1

batches_mean = torch.empty(size = (n_up, 0, 60, 60))
batches_covariance = torch.empty(size = (n_up, 0, 60**2, 60**2))
batches_lml = torch.empty(size = (n_up, 0))
batches_rmse = torch.empty(size = (n_up, 0, 1))

for batch in dataloader:
    srgp = SuperresolutionGP(train_tensor = batch, scaling_bool = False)
    mean, covar, lml = srgp.predict_and_evaluate_training()
    rmse = srgp.rmse(srgp.train_bed_ground_truth, mean.unsqueeze(2)) # mean needs explicit channel dimension

    batches_mean = torch.cat((batches_mean, mean), dim = 1)
    batches_covariance = torch.cat((batches_covariance, covar), dim = 1)
    batches_lml = torch.cat((batches_lml, lml), dim = 1)
    batches_rmse = torch.cat((batches_rmse, rmse), dim = 1)

In [ ]:
# 15 sec for 10 images so 7 mins for 300 images
torch.mean(batches_rmse)

In [ ]:
batches_mean.shape

In [ ]:
srgp = SuperresolutionGP(train_tensor = training_tensor)

# mean RMSE per up factor
torch.mean(srgp.train_baseline_rmse, dim = 1)

mean, covar, lml = srgp.predict_and_evaluate_training()

for 300 images: on GPU 

In [ ]:
# srgp.rmse(ground_truth = srgp., mean)

In [ ]:
srGP = SuperresolutionGP(train_tensor = training_tensor[0:20, :, :, :])

print(srGP.pixel_base_covariance)

## Optimize hyperparameters

In [ ]:
# Initialise on subset
srGP = SuperresolutionGP(train_tensor = training_tensor[0:20, :, :, :])

In [ ]:
from scipy.optimize import minimize

def function(hyperparameters, srGP):
    srGP.set_hypers(hyperparameters)
    # Disregard upfactors_n_mean and upfactors_n_covariance. Only focus on lml
    _, _, upfactors_n_lml = srGP.predict_and_evaluate_training()
    # mean lml over all training scenes and 5 different upscaling factors
    nlml_mean = - torch.mean(upfactors_n_lml)
    print("Lambda_s: {}, Lambda_p: {}, Sigma_f: {} - LML : {}".format(np.round(hyperparameters[0], 3), np.round(hyperparameters[1], 3),np.round(hyperparameters[2], 3), np.round(nlml_mean.item(), 1)))
    return nlml_mean

# Initial variables: lambda_s, lambda_p, sigma_f (based on previous runs)
x0 = np.array([1.0, 1.0, 0.2])
bnds = ((0.1, 2.0), (0, 10.0), (0.1, 2.0))

# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize
res = minimize(function, 
                   x0, 
                   method = 'Nelder-Mead', 
                   args = (srGP), # extra arguments passed into 
                   bounds = bnds, # avoid issues with Cholesky 
                   options = {'xatol': 1e-10, 'disp': True}) #https://docs.scipy.org/doc/scipy/reference/optimize.minimize-neldermead.html#optimize-minimize-neldermead

print(res)

In [ ]:
res.x

print("Optimal signal variance: "+str(res.x[0]))
print("Optimal lengthscale: "+str(res.x[1]))
print("Optimal noise std: "+str(res.x[2]))

_LinAlgError: linalg.cholesky: The factorization could not be completed because the input is not positive-definite (the leading minor of order 1 is not positive-definite).

Cholesky error for:
0.69555556 0.88666667 0.2   
1.04333333 1.01333333 0.08   (too small sigma?)

Good performances:
0.66 0.88 0.36
2.0 2.0 0.1